In [72]:
# Imports section
import pandas as pd
import numpy as np

## Part 1. Loading the dataset

In [73]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

In [74]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [75]:
df = pd.read_csv('/content/drive/MyDrive/Fall 2024/CS 383/Assignment 4/science_data_large.csv')
display(df.head(15))

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


In [76]:
print(df.shape)

(1000, 3)


In [77]:
display(df.describe())

,Temperature °C,Mols KCL,Size nm^3
count,1000.000000,1000.000000,1.000000e+03
mean,500.500000,471.530000,5.086111e+05
std,288.819436,288.482872,4.474838e+05
min,1.000000,1.000000,1.611429e+01
25%,250.750000,226.750000,1.298267e+05
50%,500.500000,459.500000,3.827182e+05
75%,750.250000,710.250000,7.603211e+05
max,1000.000000,1000.000000,1.972127e+06


- The given data set has 2 features which are numerical and a label which is Size.
- It has 1000 rows and 3 columns.
- The above function including 'describe' displays the different statistics of the dataset and is pretty descriptive.

## Part 2. Splitting the dataset

In [78]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[['Temperature °C', 'Mols KCL']]
y = df['Size nm^3']

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 42)
print('X_train shape: ', X_train.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape: ', y_train.shape)
print('y_test shape: ', y_test.shape)

X_train shape:  (900, 2)
X_test shape:  (100, 2)
y_train shape:  (900,)
y_test shape:  (100,)


## Part 3. Perform a Linear Regression

In [81]:
# Use sklearn to train a model on the training set
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample = pd.DataFrame([[550, 480]], columns=['Temperature °C', 'Mols KCL'])
predicted_size = model.predict(sample)
print('Predicted Size: ', predicted_size)

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print('Score: ', score)

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print('Coefficients: ', coefficients)
print('Intercept: ', intercept)

Predicted Size:  [562682.67968433]
Score:  0.8552472077276095
Coefficients:  [ 866.14641337 1032.69506649]
Intercept:  -409391.47958340764


- The score of the linear regression model is 0.855. This means the model can explain about 85.5% of the variability in eruption size based on temperature and the amount of KCL.

- A score of 0.855 indicates a strong relationship between the inputs (temperature and mols KCL) and the output (size). The higher the score, the better the model is at making accurate predictions.

- $$
h(x) = -409391.48 + 866.15 \cdot \text{Temperature °C} + 1032.70 \cdot \text{Mols KCL}
$$


Sample equation: $E = mc^2$

## Part 4. Use Cross Validation

In [82]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5)
print('Cross Validation Scores: ', scores)

# Report on their finding and their significance
mean_score = np.mean(scores)
print('Mean Score: ', mean_score)


Cross Validation Scores:  [0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
Mean Score:  0.8568167899144437


The cross-validation scores obtained from the linear regression model represent  values for the model when tested on different subsets of the data. The average is approximately 0.856, which indicates that the model consistently explains about 85.6% of the variance in size across the various data splits. This suggests that the model performs well and is robust against overfitting.

## Part 5. Using Polynomial Regression

In [71]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree = 2)
X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)
model = LinearRegression()
model.fit(X_poly_train, y_train)

sample_poly = poly.transform([[550, 480]])
predicted_size_poly = model.predict(sample_poly)
print('Predicted Size;', predicted_size_poly)

score_poly = model.score(X_poly_test, y_test)
print('Polynomial Regression Score:', score_poly)

coefficients_poly = model.coef_
intercept_poly = model.intercept_
print('Polynomial Coefficients:', coefficients_poly)
print('Polynomial Intercept:', intercept_poly)


# Report on the metrics and output the resultant equation as you did in Part 3.


Predicted Size; [541182.85713]
Polynomial Regression Score: 1.0
Polynomial Coefficients: [ 0.00000000e+00  1.20000000e+01 -1.27195488e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02]
Polynomial Intercept: 2.0477105863392353e-05


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(


$$
\text{h(x)} = 12(\text{Temperature °C}) + 2(\text{Temperature °C})(\text{Mols KCl}) + 0.02857(\text{Mols KCl})^2
$$



In this part, we applied a polynomial regression model of degree 2 to predict size based on temperature (°C) and moles of KCL. The model achieved a perfect score of 1.0, indicating it explains 100% of the variance in size, though this raises high concerns about potential overfitting. For a sample input of 550°C and 480 moles of KCL, the model predicted a size of 541,182.86. The coefficients from the model reveal that temperature has a strong positive effect on eruption size, both linearly and quadratically, while the impact of KCL is smaller but still significant.